# Incident Investigation Playbook

Run a guided investigation starting from a live alert.

In [ ]:
import os
from IPython.display import Markdown, display
from clario360 import Client

sdk = Client.from_env()
alert_id = os.environ.get('CLARIO360_ALERT_ID', '')
if not alert_id:
    alerts = sdk.cyber.alerts.list(severity='critical,high', per_page=1, sort='created_at', order='desc')
    if not alerts.data:
        raise RuntimeError('No recent alerts available. Set CLARIO360_ALERT_ID to investigate a specific case.')
    alert_id = alerts.data[0].id
alert = sdk.cyber.alerts.get(alert_id)
print(f'Investigating {alert.title}')

## Collect context

Asset, vulnerability, and related-alert lookups are all performed through the authenticated SDK.

In [ ]:
asset_summaries = []
for asset_summary in getattr(alert, 'affected_assets', []):
    asset = sdk.cyber.assets.get(asset_summary.id)
    vulns = sdk.cyber.assets.vulnerabilities(asset.id)
    asset_summaries.append({
        'name': asset.name,
        'type': asset.type,
        'criticality': getattr(asset, 'criticality', 'unknown'),
        'open_vulnerabilities': vulns.meta.total,
    })
related = sdk.cyber.alerts.related(alert.id)
asset_summaries

## Produce the investigation summary

The final markdown block can be copied into a case record or executive update.

In [ ]:
summary = f"""
# Investigation Summary: {alert.title}

- Alert ID: {alert.id}
- Severity: {alert.severity}
- Status: {alert.status}
- Confidence: {getattr(alert, 'confidence_score', 0):.0%}

## Related alerts
{chr(10).join(f'- {item.title}' for item in getattr(related, 'data', [])) or '- None'}

## Assets
{chr(10).join(f"- {item['name']} ({item['type']}) - {item['open_vulnerabilities']} open vulns" for item in asset_summaries) or '- None'}
"""
display(Markdown(summary))